In [58]:
## IMPORTING REQUIRED LIBRARIES:-


import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec

from sklearn.metrics.pairwise import cosine_similarity

In [59]:
df = pd.read_csv("Dataset .csv")
df.head()

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,...,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",...,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314
1,6304287,Izakaya Kikufuji,162,Makati City,"Little Tokyo, 2277 Chino Roces Avenue, Legaspi...","Little Tokyo, Legaspi Village, Makati City","Little Tokyo, Legaspi Village, Makati City, Ma...",121.014101,14.553708,Japanese,...,Botswana Pula(P),Yes,No,No,No,3,4.5,Dark Green,Excellent,591
2,6300002,Heat - Edsa Shangri-La,162,Mandaluyong City,"Edsa Shangri-La, 1 Garden Way, Ortigas, Mandal...","Edsa Shangri-La, Ortigas, Mandaluyong City","Edsa Shangri-La, Ortigas, Mandaluyong City, Ma...",121.056831,14.581404,"Seafood, Asian, Filipino, Indian",...,Botswana Pula(P),Yes,No,No,No,4,4.4,Green,Very Good,270
3,6318506,Ooma,162,Mandaluyong City,"Third Floor, Mega Fashion Hall, SM Megamall, O...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.056475,14.585318,"Japanese, Sushi",...,Botswana Pula(P),No,No,No,No,4,4.9,Dark Green,Excellent,365
4,6314302,Sambo Kojin,162,Mandaluyong City,"Third Floor, Mega Atrium, SM Megamall, Ortigas...","SM Megamall, Ortigas, Mandaluyong City","SM Megamall, Ortigas, Mandaluyong City, Mandal...",121.057508,14.584450,"Japanese, Korean",...,Botswana Pula(P),Yes,No,No,No,4,4.8,Dark Green,Excellent,229


In [60]:
print("No of rows:",df.shape[0])
print("No of rows:",df.shape[1])

No of rows: 9551
No of rows: 21


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   object 
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   object 
 4   Address               9551 non-null   object 
 5   Locality              9551 non-null   object 
 6   Locality Verbose      9551 non-null   object 
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   object 
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   object 
 12  Has Table booking     9551 non-null   object 
 13  Has Online delivery   9551 non-null   object 
 14  Is delivering now     9551 non-null   object 
 15  Switch to order menu 

In [62]:
df.describe()

,Restaurant ID,Country Code,Longitude,Latitude,Average Cost for two,Price range,Aggregate rating,Votes
count,9.551000e+03,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000,9551.000000
mean,9.051128e+06,18.365616,64.126574,25.854381,1199.210763,1.804837,2.666370,156.909748
std,8.791521e+06,56.750546,41.467058,11.007935,16121.183073,0.905609,1.516378,430.169145
min,5.300000e+01,1.000000,-157.948486,-41.330428,0.000000,1.000000,0.000000,0.000000
25%,3.019625e+05,1.000000,77.081343,28.478713,250.000000,1.000000,2.500000,5.000000
50%,6.004089e+06,1.000000,77.191964,28.570469,400.000000,2.000000,3.200000,31.000000
75%,1.835229e+07,1.000000,77.282006,28.642758,700.000000,2.000000,3.700000,131.000000
max,1.850065e+07,216.000000,174.832089,55.976980,800000.000000,4.000000,4.900000,10934.000000


In [64]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [65]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
9546    False
9547    False
9548    False
9549    False
9550    False
Length: 9551, dtype: bool

In [66]:
df.columns

Index(['Restaurant ID', 'Restaurant Name', 'Country Code', 'City', 'Address',
       'Locality', 'Locality Verbose', 'Longitude', 'Latitude', 'Cuisines',
       'Average Cost for two', 'Currency', 'Has Table booking',
       'Has Online delivery', 'Is delivering now', 'Switch to order menu',
       'Price range', 'Aggregate rating', 'Rating color', 'Rating text',
       'Votes'],
      dtype='object')

In [67]:
columns = ["Restaurant Name","Cuisines","Price range","Aggregate rating"]


In [68]:
new_df = df[columns]
new_df

,Restaurant Name,Cuisines,Price range,Aggregate rating
0,Le Petit Souffle,"French, Japanese, Desserts",3,4.8
1,Izakaya Kikufuji,Japanese,3,4.5
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4
3,Ooma,"Japanese, Sushi",4,4.9
4,Sambo Kojin,"Japanese, Korean",4,4.8
...,...,...,...,...
9546,Naml۱ Gurme,Turkish,3,4.1
9547,Ceviz A��ac۱,"World Cuisine, Patisserie, Cafe",3,4.2
9548,Huqqa,"Italian, World Cuisine",4,3.7
9549,A���k Kahve,Restaurant Cafe,4,4.0


In [69]:
new_df.isnull().sum()

Restaurant Name     0
Cuisines            9
Price range         0
Aggregate rating    0
dtype: int64

In [71]:
new_df = new_df.dropna()


In [72]:
new_df = new_df.drop_duplicates()

In [73]:
df["Price range"].value_counts()

Price range
1    4444
2    3113
3    1408
4     586
Name: count, dtype: int64

In [74]:
# ==========================================================
# Convert Price Range into Labels
# ==========================================================

price_mapping = {
    1: "budget",
    2: "moderate",
    3: "expensive",
    4: "luxury"
}

new_df['Price Category'] = new_df['Price range'].map(price_mapping)



In [75]:
new_df

,Restaurant Name,Cuisines,Price range,Aggregate rating,Price Category
0,Le Petit Souffle,"French, Japanese, Desserts",3,4.8,expensive
1,Izakaya Kikufuji,Japanese,3,4.5,expensive
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4,luxury
3,Ooma,"Japanese, Sushi",4,4.9,luxury
4,Sambo Kojin,"Japanese, Korean",4,4.8,luxury
...,...,...,...,...,...
9546,Naml۱ Gurme,Turkish,3,4.1,expensive
9547,Ceviz A��ac۱,"World Cuisine, Patisserie, Cafe",3,4.2,expensive
9548,Huqqa,"Italian, World Cuisine",4,3.7,luxury
9549,A���k Kahve,Restaurant Cafe,4,4.0,luxury


In [76]:
# ==========================================================
# Create Rating Category
# ==========================================================

new_df['Rating Category'] = (
    new_df['Aggregate rating']
      .round()
      .astype(int)
      .apply(lambda x: f"rating_{x}")
)



In [77]:
new_df

,Restaurant Name,Cuisines,Price range,Aggregate rating,Price Category,Rating Category
0,Le Petit Souffle,"French, Japanese, Desserts",3,4.8,expensive,rating_5
1,Izakaya Kikufuji,Japanese,3,4.5,expensive,rating_4
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4,luxury,rating_4
3,Ooma,"Japanese, Sushi",4,4.9,luxury,rating_5
4,Sambo Kojin,"Japanese, Korean",4,4.8,luxury,rating_5
...,...,...,...,...,...,...
9546,Naml۱ Gurme,Turkish,3,4.1,expensive,rating_4
9547,Ceviz A��ac۱,"World Cuisine, Patisserie, Cafe",3,4.2,expensive,rating_4
9548,Huqqa,"Italian, World Cuisine",4,3.7,luxury,rating_4
9549,A���k Kahve,Restaurant Cafe,4,4.0,luxury,rating_4


In [78]:
def preprocess_text(text):
    text = text.lower()
    cuisines = text.split(",")
    cuisines = [c.strip().replace(" ","_") for c in cuisines]
    return cuisines


new_df['Cuisines Tokens'] = new_df['Cuisines'].apply(preprocess_text)
new_df.head()

,Restaurant Name,Cuisines,Price range,Aggregate rating,Price Category,Rating Category,Cuisines Tokens
0,Le Petit Souffle,"French, Japanese, Desserts",3,4.8,expensive,rating_5,"[french, japanese, desserts]"
1,Izakaya Kikufuji,Japanese,3,4.5,expensive,rating_4,[japanese]
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4,luxury,rating_4,"[seafood, asian, filipino, indian]"
3,Ooma,"Japanese, Sushi",4,4.9,luxury,rating_5,"[japanese, sushi]"
4,Sambo Kojin,"Japanese, Korean",4,4.8,luxury,rating_5,"[japanese, korean]"


In [81]:
# ==========================================================
# Add Price and Rating Tokens
# ==========================================================

new_df['Restaurant Tokens'] = new_df.apply(
    lambda row: row['Cuisines Tokens'] + [row['Price Category'], row['Rating Category']],
    axis=1
)



In [83]:
new_df

,Restaurant Name,Cuisines,Price range,Aggregate rating,Price Category,Rating Category,Cuisines Tokens,Restaurant Tokens
0,Le Petit Souffle,"French, Japanese, Desserts",3,4.8,expensive,rating_5,"[french, japanese, desserts]","[french, japanese, desserts, expensive, rating_5]"
1,Izakaya Kikufuji,Japanese,3,4.5,expensive,rating_4,[japanese],"[japanese, expensive, rating_4]"
2,Heat - Edsa Shangri-La,"Seafood, Asian, Filipino, Indian",4,4.4,luxury,rating_4,"[seafood, asian, filipino, indian]","[seafood, asian, filipino, indian, luxury, rat..."
3,Ooma,"Japanese, Sushi",4,4.9,luxury,rating_5,"[japanese, sushi]","[japanese, sushi, luxury, rating_5]"
4,Sambo Kojin,"Japanese, Korean",4,4.8,luxury,rating_5,"[japanese, korean]","[japanese, korean, luxury, rating_5]"
...,...,...,...,...,...,...,...,...
9546,Naml۱ Gurme,Turkish,3,4.1,expensive,rating_4,[turkish],"[turkish, expensive, rating_4]"
9547,Ceviz A��ac۱,"World Cuisine, Patisserie, Cafe",3,4.2,expensive,rating_4,"[world_cuisine, patisserie, cafe]","[world_cuisine, patisserie, cafe, expensive, r..."
9548,Huqqa,"Italian, World Cuisine",4,3.7,luxury,rating_4,"[italian, world_cuisine]","[italian, world_cuisine, luxury, rating_4]"
9549,A���k Kahve,Restaurant Cafe,4,4.0,luxury,rating_4,[restaurant_cafe],"[restaurant_cafe, luxury, rating_4]"


In [87]:
# ==========================================================
# Step 7: Train the Word2Vec Model
# ==========================================================

from gensim.models import Word2Vec

word2vec_model = Word2Vec(
    sentences=new_df['Restaurant Tokens'],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

In [88]:
print(word2vec_model.wv.index_to_key)

['rating_3', 'rating_4', 'north_indian', 'budget', 'moderate', 'chinese', 'expensive', 'fast_food', 'mughlai', 'italian', 'continental', 'luxury', 'cafe', 'bakery', 'desserts', 'south_indian', 'street_food', 'american', 'rating_2', 'pizza', 'mithai', 'asian', 'thai', 'rating_5', 'burger', 'mexican', 'seafood', 'beverages', 'european', 'japanese', 'ice_cream', 'biryani', 'mediterranean', 'finger_food', 'healthy_food', 'sushi', 'indian', 'steak', 'lebanese', 'sandwich', 'salad', 'breakfast', 'raw_meats', 'bar_food', 'bbq', 'tibetan', 'tea', 'french', 'hyderabadi', 'bengali', 'southern', 'vegetarian', 'juices', 'arabian', 'brazilian', 'kerala', 'malaysian', 'vietnamese', 'international', 'grill', 'goan', 'middle_eastern', 'korean', 'tapas', 'coffee_and_tea', 'rajasthani', 'tex-mex', 'kashmiri', 'british', 'spanish', 'modern_indian', 'turkish', 'greek', 'indonesian', 'pakistani', 'modern_australian', 'latin_american', 'kebab', 'andhra', 'chettinad', 'maharashtrian', 'gujarati', 'western', 

In [86]:
new_df = new_df[new_df['Aggregate rating'] > 0].reset_index(drop=True)

In [89]:
# ==========================================================
# Function to Generate Restaurant Vector
# ==========================================================

def get_restaurant_vector(tokens, model):
    
    vectors = []

    for token in tokens:
        if token in model.wv:
            vectors.append(model.wv[token])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [90]:
# ==========================================================
# Generate Restaurant Embeddings
# ==========================================================

new_df['Restaurant Vector'] = new_df['Restaurant Tokens'].apply(
    lambda tokens: get_restaurant_vector(tokens, word2vec_model)
)

In [91]:
new_df[['Restaurant Name', 'Restaurant Vector']].head()

,Restaurant Name,Restaurant Vector
0,Le Petit Souffle,"[-0.05613504, -0.041592047, -0.007869039, -0.1..."
1,Izakaya Kikufuji,"[-0.06299284, -0.043021347, 0.009537119, -0.15..."
2,Heat - Edsa Shangri-La,"[-0.06024318, -0.04433601, -0.0036505887, -0.1..."
3,Ooma,"[-0.07871345, -0.05076428, -0.003885539, -0.16..."
4,Sambo Kojin,"[-0.068860136, -0.04475134, -0.004425205, -0.1..."


In [95]:
restaurant_vectors = np.vstack(new_df['Restaurant Vector'].values)

In [96]:
similarity_matrix = cosine_similarity(restaurant_vectors)

print(similarity_matrix.shape)

(6875, 6875)


In [99]:
# ==========================================================
# Recommendation Function Based on User Preferences
# ==========================================================

def recommend_restaurants(cuisine, price_category, rating_category, top_n=5):

    # Create user tokens
    user_tokens = [
        cuisine.lower().replace(" ", "_"),
        price_category.lower(),
        rating_category.lower()
    ]

    # Generate user preference vector
    user_vector = get_restaurant_vector(user_tokens, word2vec_model)

    # Compute similarity with all restaurant vectors
    similarity_scores = cosine_similarity(
        [user_vector],
        restaurant_vectors
    )[0]

    # Get indices of top similar restaurants
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # Create result DataFrame
    recommendations = new_df.iloc[top_indices][
        ['Restaurant Name', 'Cuisines', 'Price Category', 'Aggregate rating']
    ].copy()

    recommendations['Similarity Score'] = similarity_scores[top_indices]

    return recommendations

In [100]:
recommend_restaurants(
    cuisine="Japanese",
    price_category="luxury",
    rating_category="rating_5"
)

,Restaurant Name,Cuisines,Price Category,Aggregate rating,Similarity Score
53,Sushi Leblon,Japanese,luxury,4.6,1.000000
4,Sambo Kojin,"Japanese, Korean",luxury,4.8,0.999793
6723,Bone Daddies,"Ramen, Japanese",expensive,4.6,0.999695
20,NIU by Vikings,"Seafood, American, Mediterranean, Japanese",luxury,4.7,0.999603
3,Ooma,"Japanese, Sushi",luxury,4.9,0.999592
